In [39]:
import numpy as np
import scipy.io

mat = scipy.io.loadmat("Example_Data.mat", simplify_cells=True)
print("Variables found:", [k for k in mat.keys() if not k.startswith("__")])

Variables found: ['X1', 'X2', 'X3', 'X4', 'dt1', 'times']


In [40]:
np.set_printoptions(threshold=np.inf, linewidth=np.inf)

In [41]:
mat

{'__header__': b'MATLAB 5.0 MAT-file, Platform: MACI64, Created on: Thu Sep 19 14:47:05 2019',
 '__version__': '1.0',
 '__globals__': [],
 'X1': array([[ 3.42908546,  3.19274123,  3.53839328,  3.31847314,  3.69705432,  2.82817993,  3.01008433,  2.92454082,  2.71791902,  3.18131067,  2.88045811,  2.34866792,  2.73032284,  2.58692595,  2.46522401,  2.98124499,  3.16088835,  3.58536949,  2.99753553,  2.24732407,  4.41721861],
        [ 1.41638522,  3.0676972 ,  3.1863934 ,  2.86228189,  3.02973054,  3.17748439,  4.07581682,  3.51979283,  2.70618893,  2.64616315,  3.37179791,  4.10344852,  3.65476354,  3.7197787 ,  2.979342  ,  3.25321735,  4.65458835,  4.62677458,  4.23685337,  4.27564477,  4.04994982],
        [10.47417432,  5.64088488,  4.27369217,  3.17409743,  2.35662506,  3.2554804 ,  3.41231532,  3.16116015,  2.62988054,  2.16487516,  3.12122632,  3.36231078,  4.39396574,  3.32801989,  3.55258272,  2.99177057,  3.70009994,  4.66984447,  4.72019031,  4.15290949,  4.3785778 ],
       

In [4]:

import numpy as np
from dataclasses import dataclass
from typing import Optional
from helpers.BINGOdata import BINGOData, BINGOParams, BINGOState

In [84]:
params = BINGOParams(nstep=4, its=3000)
data = BINGOData(ts=[mat['X1'], mat['X2']], Tsam=[mat["dt1"], mat['times']], params=params)

In [79]:

def bingo_init(data: BINGOData, params: BINGOParams, scale: bool = True) -> tuple[BINGOData, BINGOState]:
    """
    Initialises BINGO. If scale=True, each gene is scaled so that its
    range across all experiments maps to [0, 1]. Modifies data.ts in place.
    Returns (data, state).
    """
    # ── Scaling ────────────────────────────────────────────────────────────────
    if scale:
        maxs = np.full(data.n_genes, -np.inf)
        mins = np.full(data.n_genes,  np.inf)
        for ts_i in data.ts:
            maxs = np.maximum(maxs, ts_i.max(axis=1))
            mins = np.minimum(mins, ts_i.min(axis=1))
        scale_factor = maxs - mins          # [n_genes]
        data.ts = [ts_i / scale_factor[:, None] for ts_i in data.ts]
        # recompute maxs/mins in scaled space
        maxs = maxs / scale_factor
        mins = maxs - 1.0
    else:
        maxs = np.full(data.n_genes, 1.0)
        mins = np.zeros(data.n_genes)

    n      = data.n_genes
    n_in   = data.input[0].shape[0] if data.input is not None else 0
    nstep  = data.params.nstep
    nr_pi  = data.params.nr_pi
    n_exp  = data.n_experiments

    # ── Ser matrix [4 x n_experiments] ────────────────────────────────────────
    # Rows 0-1: start/end indices in the coarse (measurement) grid (0-based)
    # Rows 2-3: start/end indices in the fine (interpolated) grid (0-based)
    Ser = np.zeros((4, n_exp), dtype=int)
    Ser[0, 0] = 0
    Ser[1, 0] = data.ts[0].shape[1] - 1
    Ser[2, 0] = 0
    Ser[3, 0] = nstep * (Ser[1, 0] - Ser[0, 0])
    for j in range(1, n_exp):
        n_tp_j = data.ts[j].shape[1]
        Ser[0, j] = Ser[1, j-1] + 1
        Ser[1, j] = Ser[0, j] + n_tp_j - 1
        Ser[2, j] = Ser[3, j-1] + 1
        Ser[3, j] = Ser[2, j] + nstep * (n_tp_j - 1)

    # ── Initial trajectory: linear interpolation between measurements ──────────
    total_fine = Ser[3, -1] + 1
    xs = np.zeros((n, total_fine))
    for j in range(n_exp):
        ts_j = data.ts[j]
        n_tp = ts_j.shape[1]
        for jj in range(n_tp - 1):
            t = np.arange(nstep) / nstep           # [0, 1/nstep, ..., (nstep-1)/nstep]
            col_start = Ser[2, j] + jj * nstep
            xs[:, col_start:col_start + nstep] = (
                ts_j[:, jj:jj+1] * (1 - t) + ts_j[:, jj+1:jj+2] * t
            )
        xs[:, Ser[3, j]] = ts_j[:, -1]

    # ── Signal statistics (used to initialise gamma and q) ────────────────────
    nry   = np.zeros(n)
    nry_q = np.zeros(n)
    Ttot  = 0.0
    for j in range(n_exp):
        ts_j   = data.ts[j]
        tsam_j = data.Tsam[j]
        dt     = tsam_j[1:] - tsam_j[:-1]
        diff2  = (ts_j[:, 1:] - ts_j[:, :-1]) ** 2
        nry   += (diff2 / dt).sum(axis=1)
        nry_q += diff2.sum(axis=1)
        Ttot  += tsam_j[-1] - tsam_j[0]
    nry   /= Ttot
    nry_q /= Ttot

    # ── Initial connectivity and hyperparameters ───────────────────────────────
    # S    = np.random.rand(n, n + n_in) > 0.9
    # For debugging purposes
    S = np.array([[1,0,0,0,0],[0,0,0,0,0],[0,0,0,0,0],[0,0,0,0,0],[1,0,0,0,0]])
    bets = np.abs(np.random.randn(n, n + n_in) * 0.5) + 1e-3
    psi  = np.vstack([
        (maxs - mins)[:, None] * np.random.rand(n, nr_pi) + mins[:, None],
        np.random.rand(n_in, nr_pi)
    ])

    state = BINGOState(
        q     = nry_q / 20,
        gamma = nry,
        r     = np.full(n, 0.0006 * 0.1),
        xs    = xs,
        P     = np.full(n, -1e8),
        bets  = bets,
        J     = np.full(n, 1e8),
        S     = S,
        psi   = psi,
        ma    = np.full(n, 0.1),
        mb    = np.full(n, 0.05),
        Ser   = Ser,
    )

    return data, state


In [89]:
data, state = bingo_init(data, params=params, scale=True)

In [81]:
state.S

array([[1, 0, 0, 0, 0],
       [0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0],
       [1, 0, 0, 0, 0]])

In [46]:
state.r.shape

(5,)

In [47]:
import time

In [ ]:
def bingo(data: BINGOData, state: BINGOState):
    """
    Main BINGO sampler.
    Returns: Plink, chain, xstore, state, stats
    """
    # ── Unpack state ───────────────────────────────────────────────────────────
    q       = state.q.copy()
    gamma   = state.gamma.copy()
    r       = state.r.copy()
    xs_old  = state.xs.copy()
    Pold    = state.P.copy()
    betsold = state.bets.copy()
    Jold    = state.J.copy()
    Sold    = state.S.copy()
    psiold  = state.psi.copy()
    ma      = state.ma.copy()
    mb      = state.mb.copy()
    Ser     = state.Ser  # [4 x n_experiments], 0-based

    # ── Basic dimensions ───────────────────────────────────────────────────────
    n     = data.n_genes
    n_in  = data.input[0].shape[0] if data.input is not None else 0
    nstep = data.params.nstep
    M     = psiold.shape[1]   # number of pseudo-inputs
    n_exp = data.n_experiments

    # ── Concatenate all time series into one matrix y ─────────────────────────
    y    = np.hstack(data.ts)                      # [n x total_timepoints]
    #print(y.shape)
    Tsam = data.Tsam                                 # list of time vectors

    # ── Range of y (and inputs) for pseudo-input bounds ───────────────────────
    rany = np.vstack([y.min(axis=1), y.max(axis=1)]).T  # [n x 2]

    # ── All genes are included (no knockouts) ─────────────────────────────────
    geneList = np.arange(n)                          # 0-based

    # ── Log prior on links ────────────────────────────────────────────────────
    if data.params.link_pr is not None:
        log_link_pr = np.full((n, n + n_in), np.log(data.params.link_pr))
    else:
        log_link_pr = np.full((n, n + n_in), -np.log(n))

    # ── Prior network (sure links) ────────────────────────────────────────────
    S_aux = np.zeros((n, n + n_in))
    if data.sure is not None:
        S_aux[:n, :n] = data.sure
    Sold = np.maximum(Sold, S_aux)
    Sold = np.minimum(Sold, 1 + S_aux)
    S_aux = 1.0 - np.abs(S_aux)                     # free entries = 1

    # ── nomiss: all ones since we have no missing data ────────────────────────
    nomiss = np.ones((n, y.shape[1]))

    # ── derind_full, yind, d_full ──────────────────────────────────────────────
    # derind_full : indices into xs_old of the "left" point of each fine interval
    # yind        : indices into xs_old corresponding to measurement timepoints
    # d_full      : fine time step for each interval, repeated nstep times

    # first experiment
    n_tp0 = Ser[1, 0] - Ser[0, 0]                   # number of intervals in exp 0
    derind_full = np.arange(nstep * n_tp0) + Ser[2, 0]
    yind        = Ser[2, 0] + np.arange(Ser[1, 0] - Ser[0, 0] + 1) * nstep
    d_full      = np.repeat(np.diff(Tsam[0]) / nstep, nstep)

    for j in range(1, n_exp):
        n_tp_j      = Ser[1, j] - Ser[0, j]
        derind_j    = np.arange(nstep * n_tp_j) + Ser[2, j]
        yind_j      = Ser[2, j] + np.arange(n_tp_j + 1) * nstep
        d_j         = np.repeat(np.diff(Tsam[j]) / nstep, nstep)
        derind_full = np.concatenate([derind_full, derind_j])
        yind        = np.concatenate([yind, yind_j])
        d_full      = np.concatenate([d_full, d_j])

    # ── Signal statistics (quadratic variation, total variation) ──────────────
    nry        = np.zeros(n)
    totvar     = np.zeros(n)
    Total_time = np.zeros(n)
    for l in range(n_exp):
        ts_l   = data.ts[l]
        tsam_l = Tsam[l]
        dt_l   = tsam_l[1:] - tsam_l[:-1]
        diff_l = ts_l[:, 1:] - ts_l[:, :-1]
        nry       += (diff_l ** 2 / dt_l).sum(axis=1)
        totvar    += np.abs(diff_l).sum(axis=1)
        Total_time += tsam_l[-1] - tsam_l[0]
    nry    /= Total_time
    totvar /= Total_time

    # ── Piecewise linear embedding matrix Pr and interpolation matrix Pintc ───
    # Pr    : [mm x max_n_tp] maps measurement values to fine grid
    # Pintc : [nstep-1 x nstep-1] sine-based interpolation for CN sampler
    max_intervals = int((Ser[3, :] - Ser[2, :]).max()) + 1
    max_tp        = int((Ser[1, :] - Ser[0, :]).max()) + 1
    Pr = np.zeros((max_intervals, max_tp))
    t  = np.arange(1, nstep + 1) / nstep               # [1/nstep ... 1]
    Pr[:nstep, 0]    = t[::-1]                          # first column
    Pr[-nstep:, -1]  = t                                # last column
    for j in range(1, max_tp - 1):
        Pr[(j-1)*nstep+1 : j*nstep+1,     j] = t
        Pr[j*nstep       : (j+1)*nstep,   j] = t[::-1]

    k_vec  = np.arange(1, nstep).reshape(-1, 1)        # [nstep-1 x 1]
    Pintc  = (np.sin(k_vec * k_vec.T / nstep * np.pi)
              / (np.pi * k_vec.T) * 2 ** 0.5)          # [nstep-1 x nstep-1]

    # ── Initialise accumulators ────────────────────────────────────────────────
    Plink    = np.zeros_like(Sold)
    chain    = 0
    acctraj  = 0
    xstore   = np.zeros_like(xs_old)
    acctop   = np.zeros(n)
    acchyp   = np.zeros(n)
    accr     = np.zeros(n)
    yold     = xs_old[:, yind]

    time_mark = time.time()
    for k in range(data.params.its):

        # ── Topology sampling ──────────────────────────────────────────────────
        for i in geneList:

            S = Sold[i, :].copy()
        
            # Decide whether to change topology or only hyperparameters
            top_change = float(np.random.rand() > 0.333)
     
            # Free entries for gene i (not forced by prior)
            inds = np.where(S_aux[i, :] > 0.5)[0]
    
            n_on = S[inds].sum()
  
            # i had too many false positives, debugging that
            # topc = float(
            #     np.random.rand() > 0.5
            #     and n_on > 0.5
            #     and n_on < len(inds) - 0.5
            # )

            topc = float(
                (np.random.rand() > 0.5)
                * (n_on > 0.5)
                * (n_on < len(inds) - 0.5)
            )
            

            # Move type 1: flip one entry
            if top_change and not topc:
                indc = np.random.randint(len(inds))
                S[inds[indc]] = 1 - S[inds[indc]]

            # Move type 2: swap a 0 and a 1
            if top_change and topc:
                ind1 = np.where(S[inds] > 0.5)[0]
                ind0 = np.where(S[inds] < 0.5)[0]
                indc01 = ind0[np.random.randint(len(ind0))]
                indc10 = ind1[np.random.randint(len(ind1))]
                S[inds[indc01]] = 1
                 [inds[indc10]] = 0

            # Sample relevance parameters
            bets  = (1 - data.params.ebeta**2)**0.5 * betsold[i, :] + data.params.ebeta * np.random.randn(n + n_in)
            beta  = 0.5 + 0.45 * bets
            p_bets = np.exp(-np.abs(beta)) / np.exp(-(beta - 0.5)**2 / (2 * 0.45**2))
            beta  = np.abs(beta)

            # Sample other hyperparameters
            gamma_tr = gamma[i] + data.params.egamma * nry[i] * np.random.randn()
            gamma_tr = 1e-4 + abs(gamma_tr - 1e-4)
            matr = ma[i] + data.params.ea * np.random.randn()
            matr = 1e-7 + abs(matr - 1e-7)
            mbtr = mb[i] + data.params.eb * np.random.randn()
            mbtr = 1e-7 + abs(mbtr - 1e-7)

            # No knockouts: derind and d are always the full arrays
            derind = derind_full
            d      = d_full
            N      = len(derind)
            #print(f'N is {N}')
            # Form covariance matrices
            KM  = np.zeros((M, M))
            KNM = np.zeros((N, M))
            for j in np.where(S[:n] > 0.5)[0]:
                diff_M  = psiold[j, :][None, :] - psiold[j, :][:, None]   # [M x M]
                diff_NM = psiold[j, :][None, :] - xs_old[j, derind][:, None]  # [N x M]
                KM  += beta[j] * diff_M**2
                KNM += beta[j] * diff_NM**2

            KM  = gamma_tr * np.exp(-KM)
            KNM = gamma_tr * np.exp(-KNM)
            #print(f'S is {S}')
            #print(f'xs wanted { xs_old[0, derind[:5]]}')
            # Compute the load
            A   = KM + (1/q[i]) * (KNM.T * d) @ KNM + 1e-5 * np.eye(M)
            KC  = np.linalg.cholesky(A).T          # upper triangular, matches MATLAB chol
            der = ((xs_old[i, derind + 1] - xs_old[i, derind])
                   - d * (mbtr - matr * xs_old[i, derind])) / q[i]
            # There was a bug here : ld  = np.linalg.solve(KC, KNM.T @ der)
            #print(f'kc shape: {KC.shape}, KNM shape:{KNM.shape}, der shape:{der.shape}')
            ld  = np.linalg.solve(KC.T, KNM.T @ der)
            # print(f'km shape is {KM.shape}, KNM shape: {KNM.shape}\n KC shape = {KC.shape}')
            # print(f'der shape {der.shape}')
            # print(f'ld shape: {ld.shape}')
            # print(f'km is {KM}')
            # print(f'knm: {KNM}')
            # print(f'ld is {ld}')
            # Wiener measure term (quadratic variation of yold)
            nrY = 0.0
            for l in range(n_exp):
                yi  = yold[i, Ser[0, l]:Ser[1, l] + 1]
                dt_l = Tsam[l][1:] - Tsam[l][:-1]
                nrY += ((yi[1:] - yi[:-1])**2 / dt_l).sum()
            #print(f'nrY is {nrY}')
            # Cost function
            J1 = (0.5 * nrY / q[i]
                  - 0.5 * ld @ ld
                  + np.log(np.diag(KC)).sum()
                  - 0.5 * np.log(np.linalg.det(KM + 1e-5 * np.eye(M)))
                  - (mbtr - matr * xs_old[i, derind]) @ (xs_old[i, derind + 1] - xs_old[i, derind]) / q[i]
                  + 0.5 / q[i] * np.sum(d * (mbtr - matr * xs_old[i, derind])**2))

            PS = (S * log_link_pr[i, :]).sum() + np.log(p_bets).sum()

            # Acceptance
            P_aux_ab    = np.exp(0.1 * (ma[i] - matr + 2*mb[i] - 2*mbtr) / totvar[i])
            g_ratio     = (gamma_tr / nry[i] * (30 - gamma_tr / nry[i])
                           / (gamma[i] / nry[i] * (30 - gamma[i] / nry[i])))
            P_aux_gamma = g_ratio * np.exp(0.2 / nry[i] * (gamma[i] - gamma_tr))
            # I do this to prevent overflow
            log_acc = (np.log(P_aux_ab) + np.log(P_aux_gamma) + (PS - Pold[i] + Jold[i] - J1) / data.params.Theur)

            if log_acc > np.log(np.random.rand()):
                Sold[i, :]  = S
                Pold[i]     = PS
                Jold[i]     = J1
                betsold[i, :] = bets
                gamma[i]    = gamma_tr
                ma[i]       = matr
                mb[i]       = mbtr
                acctop[i]  += top_change
                acchyp[i]  += 1

            # Sample measurement noise variance r[i]
            rtr = r[i] + data.params.er * np.random.randn()
            rtr = 1e-8 + abs(rtr - 1e-8)
            obs = np.where(nomiss[i, :] > 0.5)[0]
            log_acc_r = (
                (1 + len(obs) / 2) * np.log(r[i] / rtr)
                + 1e-5 / r[i] - 1e-5 / rtr
                + 0.5 * (1/r[i] - 1/rtr) * np.sum((y[i, obs] - yold[i, obs])**2)
            )
            if log_acc_r > np.log(np.random.rand()):
                r[i]      = rtr
                accr[i]  += 1

                
                
        #gene loop ends
        # Trajectory sampling

        qtr = q + data.params.eq * np.random.randn(n)
        qtr = 0.5e-5 + np.abs(qtr - 0.5e-5)

        # ── Sample trajectory (no missing data) ────────────────────────────────
        yhat = np.zeros_like(yold)
        xs   = np.zeros_like(xs_old)

        for l in range(n_exp):
            c0 = Ser[0, l]               # coarse start index
            c1 = Ser[1, l]               # coarse end index
            f0 = Ser[2, l]               # fine start index
            f1 = Ser[3, l]               # fine end index
            n_tp_l = c1 - c0 + 1         # number of measurement points in exp l

            # Propose new measurement-level trajectory
            yhat[:, c0:c1+1] = (
                y[:, c0:c1+1]
                + (1 - data.params.etraj**2)**0.5 * (yold[:, c0:c1+1] - y[:, c0:c1+1])
                + data.params.etraj * np.diag(r**0.5) @ np.random.randn(n, n_tp_l)
            )

            # Propose new fine-grid trajectory
            scale = (qtr / q)**0.5                          # [n]
            cn_det = (1 - data.params.etraj**2)**0.5

            xs[:, f0:f1+1] = (
                np.diag(scale) @ (cn_det * xs_old[:, f0:f1+1])
                + (yhat[:, c0:c1+1] - np.diag(scale) @ (cn_det * yold[:, c0:c1+1]))
                @ Pr[:f1-f0+1, :n_tp_l].T
            )

            # Add Brownian bridge noise (CN sampler innovation)
            n_intervals = c1 - c0                           # number of intervals
            d_l = d_full[f0-l : f1-l]                      # fine steps for this exp
            noise = Pintc @ np.random.randn(nstep - 1, n * n_intervals)
            noise = np.vstack([noise, np.zeros((1, n * n_intervals))])
            noise = noise.reshape(-1, n).T                  # [n x n_fine_inner]

            xs[:, f0+1:f1+1] += (
                data.params.etraj
                * np.diag(qtr**0.5)
                @ ((nstep * d_l)**0.5 * noise)
            )

        # ── Sample pseudo-inputs (mirror to data box) ──────────────────────────
        psin = psiold + 0.025 * np.random.randn(*psiold.shape)
        psin = np.minimum(psin, 2 * rany[:, 1:2] - psin)
        psin = np.maximum(psin, 2 * rany[:, 0:1] - psin)
        #print(f'psin shape: {psin.shape}')
        # ── Cost function for new trajectory ──────────────────────────────────
        J1   = np.zeros(n)
        beta = np.abs(0.5 + 0.45 * betsold)                # [n x n+n_in]

        for i in geneList:
            derind = derind_full
            d      = d_full
            N      = len(derind)

            KM  = np.zeros((M, M))
            KNM = np.zeros((N, M))
            for j in np.where(Sold[i, :n] > 0.5)[0]:
                diff_M  = psin[j, :][None, :] - psin[j, :][:, None]
                diff_NM = psin[j, :][None, :] - xs[j, derind][:, None]
                KM  += beta[i, j] * diff_M**2
                KNM += beta[i, j] * diff_NM**2

            KM  = gamma[i] * np.exp(-KM)
            KNM = gamma[i] * np.exp(-KNM)

            A   = KM + (1/qtr[i]) * (KNM.T * d) @ KNM + 1e-5 * np.eye(M)
            KC  = np.linalg.cholesky(A).T
            der = ((xs[i, derind + 1] - xs[i, derind])
                   - d * (mb[i] - ma[i] * xs[i, derind])) / qtr[i]
            ld  = np.linalg.solve(KC.T, KNM.T @ der)

            nrY = 0.0
            for l in range(n_exp):
                yi   = yhat[i, Ser[0, l]:Ser[1, l] + 1]
                dt_l = Tsam[l][1:] - Tsam[l][:-1]
                nrY += ((yi[1:] - yi[:-1])**2 / dt_l).sum()

            J1[i] = (
                0.5 * nrY / qtr[i]
                - 0.5 * ld @ ld
                + np.log(np.diag(KC)).sum()
                - 0.5 * np.log(np.linalg.det(KM + 1e-5 * np.eye(M)))
                - (mb[i] - ma[i] * xs[i, derind]) @ (xs[i, derind+1] - xs[i, derind]) / qtr[i]
                + 0.5 / qtr[i] * np.sum(d * (mb[i] - ma[i] * xs[i, derind])**2)
            )

        # ── Accept or reject trajectory proposal ───────────────────────────────
        log_P_aux_q = np.sum(
            1e-5/q - 1e-5/qtr
            + (np.log(q) - np.log(qtr)) * (1.001 + 0.5 * (y.shape[1] - n_exp))
        )

        if log_P_aux_q + np.sum(Jold - J1) > np.log(np.random.rand()):
            Jold    = J1
            q       = qtr
            acctraj += 1
            xs_old  = xs
            yold    = yhat
            psiold  = psin

    
        # ── Progress reporting ─────────────────────────────────────────────────
        if k == 99:
            elapsed = time.time() - time_mark
            left    = elapsed * (data.params.its - k) / 100
            if left > 900:
                print(f"NOTE! Estimated time remaining: "
                      f"{int(left//3600)}h {int((left%3600)//60)}min")
        if k % 100 == 0:
            print(f"k={k} | "
                f"q: {q} | "
                f"gamma: {gamma} | "
                f"Jold: {Jold} | "
                f"xs nan: {np.isnan(xs_old).any()} | "
                f"xs inf: {np.isinf(xs_old).any()} | "
                f"psi nan: {np.isnan(psiold).any()}")
        # ── Thinning: store every 10th sample ──────────────────────────────────
        if k % 10 == 0:
            chain  += 1
            Plink  += Sold
            xstore += xs_old

            if k % 10000 == 0 and k > 0:
                elapsed = time.time() - time_mark
                left    = elapsed * (data.params.its - k) / 10000
                print(f"Iteration {k} | Remaining: "
                      f"{int(left//3600)}h {int((left%3600)//60)}min "
                      f"{int(left%60)}sec")
                time_mark = time.time()
        
    # ── Finalise ───────────────────────────────────────────────────────────────
    xstore /= (data.params.its / 10)

    state.q     = q
    state.gamma = gamma
    state.r     = r
    state.xs    = xs_old
    state.P     = Pold
    state.bets  = betsold
    state.J     = Jold
    state.S     = Sold
    state.psi   = psiold
    state.ma    = ma
    state.mb    = mb

    stats = {
        "acctraj": acctraj,
        "acctop":  acctop,
        "acchyp":  acchyp,
        "accr":    accr,
    }

    return Plink, chain, xstore, state, stats
    

In [91]:
Plink, chain, xstore, state, stats = bingo(data, state)

k=0 | q: [0.00186913 0.00131889 0.00140015 0.00236902 0.0013316 ] | gamma: [0.0654539  0.0499961  0.04612343 0.09164826 0.03150089] | Jold: [335.76825736 277.94567704 373.1077515  356.432807   218.37987388] | xs nan: False | xs inf: False | psi nan: False
k=100 | q: [0.00561778 0.00454516 0.00449756 0.00809376 0.00278021] | gamma: [0.06280113 0.06704903 0.08690412 0.17283265 0.06388874] | Jold: [106.84466928  64.57992093  53.04214639  76.65992864  49.89811352] | xs nan: False | xs inf: False | psi nan: False
k=200 | q: [0.008792   0.00816315 0.00936595 0.01227948 0.00569447] | gamma: [0.05321848 0.10647727 0.10086818 0.23934029 0.07082228] | Jold: [69.61375765 39.72102677 27.44034445 48.94962257 26.68614623] | xs nan: False | xs inf: False | psi nan: False
k=300 | q: [0.01592645 0.01246179 0.00722916 0.01371664 0.00739005] | gamma: [0.05474832 0.1312895  0.09178031 0.20848945 0.09323347] | Jold: [39.04732557 25.70953069 25.60536772 39.66349957 17.59617323] | xs nan: False | xs inf: Fal

In [73]:
print(stats['acchyp'] / params.its)   # should be ~0.2-0.4 typically
print(stats['acctop'] / params.its)

[0.47       0.45       0.40333333 0.42       0.37      ]
[0.15333333 0.13333333 0.14       0.11       0.08      ]


In [74]:
print(f'plink: {Plink}, chain: {chain}, xstore: {xstore}, stats: {stats}')

plink: [[ 5.  3.  3.  6. 25.]
 [29. 29.  9.  7.  1.]
 [ 1. 29. 29. 12. 26.]
 [ 4. 11. 28. 27. 10.]
 [ 4. 29.  4.  6. 29.]], chain: 30, xstore: [[0.72095241 0.69363026 0.69841951 0.72215638 0.64463841 0.68212624 0.69692812 0.74836116 0.72266119 0.72892962 0.73136153 0.7133669  0.70090156 0.72052069 0.72657116 0.74085073 0.75695827 0.72972847 0.66174966 0.64425316 0.60021604 0.59479865 0.61016143 0.65569811 0.61864572 0.62677959 0.61227405 0.62599692 0.59535245 0.60868512 0.60114625 0.58850906 0.58295545 0.57553999 0.62121507 0.60475598 0.67326971 0.6515961  0.60936228 0.63015676 0.60176337 0.5774478  0.54346859 0.54929204 0.49691661 0.50847374 0.52463793 0.54817472 0.56230864 0.57247945 0.5560483  0.53865613 0.55813444 0.53046142 0.53983118 0.49745489 0.52413744 0.5258811  0.53790513 0.59144157 0.61851876 0.63477533 0.64465226 0.65452919 0.66440611 0.68667543 0.70894475 0.73121407 0.75348339 0.72280749 0.69213159 0.66145569 0.63077979 0.5912132  0.55164661 0.51208002 0.47251343 0.586193

In [75]:
Plink/chain

array([[0.16666667, 0.1       , 0.1       , 0.2       , 0.83333333],
       [0.96666667, 0.96666667, 0.3       , 0.23333333, 0.03333333],
       [0.03333333, 0.96666667, 0.96666667, 0.4       , 0.86666667],
       [0.13333333, 0.36666667, 0.93333333, 0.9       , 0.33333333],
       [0.13333333, 0.96666667, 0.13333333, 0.2       , 0.96666667]])

In [53]:
data.params.its = 10000
Plink, chain, xstore, state, stats = bingo(data, state)
print(f'plink: {Plink}, chain: {chain}, xstore: {xstore}, stats: {stats}')

nrY is 1.401866820319371
nrY is 1.108537969351069
nrY is 2.5841224201644404
nrY is 3.967301500722238
nrY is 0.9060129287998326
nrY is 1.398801427709302
nrY is 1.1118683051102582
nrY is 2.605580066310864
nrY is 3.985733861267071
nrY is 0.9135911858719767
nrY is 1.4009268944276587
nrY is 1.1108964462703474
nrY is 2.5888552434855643
nrY is 3.9663420828761575
nrY is 0.9054693657059449
nrY is 1.3921732451450897
nrY is 1.1059198492409388
nrY is 2.5910514268375993
nrY is 3.9397978431737677
nrY is 0.9123194007024583
nrY is 1.3984362847031742
nrY is 1.1058656600598211
nrY is 2.592423085031669
nrY is 3.9474921868943977
nrY is 0.9256150502228706
nrY is 1.406798026703297
nrY is 1.1160722277637418
nrY is 2.6036559332019427
nrY is 3.979731889737476
nrY is 0.9005928395510057
nrY is 1.4025528143768948
nrY is 1.1105299665975767
nrY is 2.6046990820704927
nrY is 3.9707875255056377
nrY is 0.9134172110226243
nrY is 1.3906805756856166
nrY is 1.103032689666142
nrY is 2.5984230144688536
nrY is 3.9617738377389

In [56]:
Plink/chain

array([[0.117, 0.095, 0.192, 0.103, 0.79 ],
       [1.   , 0.   , 0.   , 0.   , 0.   ],
       [1.   , 1.   , 1.   , 1.   , 1.   ],
       [1.   , 1.   , 1.   , 1.   , 0.   ],
       [0.   , 1.   , 1.   , 1.   , 1.   ]])

In [76]:
n = data.n_genes
print(f"Acceptance probability for trajectory and q: {stats['acctraj'] / params.its * 100:.1f}")
print(f"Average/minimum acceptance probability for topology: "
      f"{stats['acctop'].mean() / params.its * 100:.1f} / "
      f"{stats['acctop'].min() / params.its * 100:.1f}")
print(f"Average/minimum acceptance probability for hyp (gamma,beta,a,b): "
      f"{stats['acchyp'].mean() / params.its * 100:.1f} / "
      f"{stats['acchyp'].min() / params.its * 100:.1f}")
print(f"Average/minimum acceptance probability for r: "
      f"{stats['accr'].mean() / params.its * 100:.1f} / "
      f"{stats['accr'].min() / params.its * 100:.1f}")

Acceptance probability for trajectory and q: 61.0
Average/minimum acceptance probability for topology: 12.3 / 8.0
Average/minimum acceptance probability for hyp (gamma,beta,a,b): 42.3 / 37.0
Average/minimum acceptance probability for r: 2.5 / 1.0


In [77]:
stats

{'acctraj': 183,
 'acctop': array([46., 40., 42., 33., 24.]),
 'acchyp': array([141., 135., 121., 126., 111.]),
 'accr': array([13.,  3.,  4., 14.,  3.])}

In [59]:
geneList

NameError: name 'geneList' is not defined